In [5]:
# .venv/bin/python3 -c "
import jax
import jax.numpy as jnp
import numpy as np
from gulps.core.jax_invariants import weyl_coordinates
from qiskit.circuit.library import SwapGate, CXGate, iSwapGate
from qiskit.quantum_info import Operator

# Check weyl_coordinates at SWAP
swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)
w = weyl_coordinates(swap)
print("SWAP weyl coords:", w)

# Check the Jacobian of weyl_coordinates w.r.t. a real perturbation
# We'll use the full try_solve pipeline's compose: params -> U -> weyl(U)
from gulps.synthesis.jax_lm import _construct_unitary, _weyl_residual, NUM_PARAMS

# Use SWAP as both prefix and basis (simplest case: identity prefix, iswap basis)
iswap = jnp.array(Operator(iSwapGate()).data, dtype=jnp.complex128)
prefix = jnp.eye(4, dtype=jnp.complex128)

# Find params where the constructed unitary is near SWAP
# The residual function: target_weyl - weyl(construct(params, prefix, basis))
target_weyl = weyl_coordinates(swap)
print("Target weyl:", target_weyl)

# Try computing the Jacobian of the Weyl residual at a point near SWAP
# Use random params
key = jax.random.PRNGKey(0)
params = jax.random.uniform(key, shape=(NUM_PARAMS,), minval=-1.0, maxval=1.0)

jac_fn = jax.jacobian(lambda p: _weyl_residual(p, prefix, iswap, target_weyl))
J = jac_fn(params)
print("Jacobian at random point (shape, rank):")
print("  shape:", J.shape)
print("  rank:", np.linalg.matrix_rank(np.array(J), tol=1e-8))
print("  singular values:", np.linalg.svd(np.array(J), compute_uv=False))

SWAP weyl coords: [1.5 0.  0.5]
Target weyl: [1.5 0.  0.5]
Jacobian at random point (shape, rank):
  shape: (3, 6)
  rank: 0
  singular values: [2.91624342e-17 1.83926131e-17 4.33449785e-18]


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from qiskit.circuit.library import iSwapGate, SwapGate, CXGate
from qiskit.quantum_info import Operator, random_unitary

from gulps.core.jax_invariants import makhlin_invariants, weyl_coordinates
from gulps.synthesis.jax_lm import _params_to_unitaries, _kron_2x2, NUM_PARAMS

iswap = jnp.array(Operator(iSwapGate()).data, dtype=jnp.complex128)
swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)

# Use iswap as BOTH prefix and basis (simulating step 2 of a decomposition)
prefix = iswap  # prefix_op = product of gates so far = one iswap
basis = iswap  # next gate

key = jax.random.PRNGKey(42)
params = jax.random.uniform(key, shape=(NUM_PARAMS,), minval=-1.0, maxval=1.0)


def construct(p):
    u0, u1 = _params_to_unitaries(p)
    return basis @ _kron_2x2(u1, u0) @ prefix


# Target: SWAP
swap_weyl = weyl_coordinates(swap)
swap_makhlin = makhlin_invariants(swap)
print("SWAP weyl:", swap_weyl)
print("SWAP makhlin:", swap_makhlin)

# Check what the constructed unitary's invariants look like
U = construct(params)
print("Constructed weyl:", weyl_coordinates(U))
print("Constructed makhlin:", makhlin_invariants(U))

# Makhlin Jacobian with real prefix
jac_m = jax.jacobian(lambda p: makhlin_invariants(construct(p)))(params)
print()
print("Makhlin Jacobian (nontrivial prefix):")
print("  rank:", np.linalg.matrix_rank(np.array(jac_m), tol=1e-8))
print("  svs:", np.linalg.svd(np.array(jac_m), compute_uv=False))

# Weyl Jacobian with real prefix
jac_w = jax.jacobian(lambda p: weyl_coordinates(construct(p)))(params)
print()
print("Weyl Jacobian (nontrivial prefix):")
print("  rank:", np.linalg.matrix_rank(np.array(jac_w), tol=1e-8))
print("  svs:", np.linalg.svd(np.array(jac_w), compute_uv=False))

# Numerical Weyl Jacobian for comparison
h = 1e-7
num_jac_w = np.zeros((3, NUM_PARAMS))
for i in range(NUM_PARAMS):
    p_plus = params.at[i].set(params[i] + h)
    p_minus = params.at[i].set(params[i] - h)
    num_jac_w[:, i] = np.array(
        (weyl_coordinates(construct(p_plus)) - weyl_coordinates(construct(p_minus)))
        / (2 * h)
    )
print()
print("Numerical Weyl Jacobian:")
print("  rank:", np.linalg.matrix_rank(num_jac_w, tol=1e-4))
print("  svs:", np.linalg.svd(num_jac_w, compute_uv=False))
print("  values:")
print(num_jac_w)

SWAP weyl: [1.5 0.  0.5]
SWAP makhlin: [-1. -0. -3.]
Constructed weyl: [6.88058479e-01 1.88268421e-01 3.46944695e-17]
Constructed makhlin: [0.21381709 0.         0.99877921]

Makhlin Jacobian (nontrivial prefix):
  rank: 2
  svs: [2.69231186e+00 2.39857074e-01 1.37914433e-17]

Weyl Jacobian (nontrivial prefix):
  rank: 2
  svs: [3.17947139e-01 3.17123817e-01 5.14405159e-17]

Numerical Weyl Jacobian:
  rank: 2
  svs: [2.65954851e+06 3.17123816e-01 2.71680102e-10]
  values:
[[ 4.74922757e-02  1.88058479e+06 -1.88058479e+06 -2.77555756e-10
   0.00000000e+00  8.32667268e-10]
 [-2.77555756e-10 -6.93889390e-10  4.16333634e-10  2.45535598e-01
   2.00643506e-01 -4.68703354e-03]
 [-6.93889390e-11  2.42861287e-10  3.46944695e-11 -1.38777878e-10
  -6.93889390e-11  1.73472348e-10]]


In [9]:
import jax.numpy as jnp
import numpy as np
from qiskit.circuit.library import SwapGate, CXGate, iSwapGate
from qiskit.quantum_info import Operator

swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)
cx = jnp.array(Operator(CXGate()).data, dtype=jnp.complex128)
iswap = jnp.array(Operator(iSwapGate()).data, dtype=jnp.complex128)

# Reproduce weyl_coordinates step by step
from gulps.core.jax_invariants import MAGIC, MAGIC_DAG, _SYSY, _M, _su4_normalize

for name, U_raw in [("SWAP", swap), ("CNOT", cx), ("iSWAP", iswap)]:
    print(f"=== {name} ===")
    U = _su4_normalize(U_raw)
    detU = jnp.linalg.det(U)
    print(f"det(U_normalized): {detU:.6f}")

    U_tilde = _SYSY @ U.T @ _SYSY
    gamma = U @ U_tilde

    ev = jnp.linalg.eigvals(gamma)
    print(f"eigenvalues of gamma: {ev}")
    print(f"|eigenvalues|: {jnp.abs(ev)}")

    two_S = jnp.angle(ev) / jnp.pi
    print(f"two_S (angle/pi): {two_S}")

    two_S = jnp.where(two_S <= -0.5, two_S + 2.0, two_S)
    print(f"two_S (after shift): {two_S}")

    S = jnp.sort(two_S / 2.0)[::-1]
    print(f"S (sorted desc): {S}")

    n = jnp.round(jnp.sum(S)).astype(jnp.int32)
    print(f"n = round(sum(S)): {n}  (sum(S) = {jnp.sum(S):.6f})")

    ones_mask = (jnp.arange(4, dtype=jnp.int32) < n).astype(jnp.float64)
    print(f"ones_mask: {ones_mask}")

    S_shifted = S - ones_mask
    print(f"S after subtract: {S_shifted}")

    S_rolled = jnp.roll(S_shifted, -n)
    print(f"S after roll(-{n}): {S_rolled}")

    c = _M @ S_rolled[:3]
    print(f"c (before folding): {c}")

    c1, c2, c3 = c[0], c[1], c[2]
    c1_folded = jnp.where(c3 < 0.0, 1.0 - c1, c1)
    c3_folded = jnp.where(c3 < 0.0, -c3, c3)
    c2_folded = jnp.maximum(c2, 0.0)
    c3_final = jnp.maximum(c3_folded, 0.0)
    print(f"FINAL: [{c1_folded}, {c2_folded}, {c3_final}]")
    print()

=== SWAP ===
det(U_normalized): 1.000000+0.000000j
eigenvalues of gamma: [2.22044605e-16-1.j 2.22044605e-16-1.j 2.22044605e-16-1.j
 2.22044605e-16-1.j]
|eigenvalues|: [1. 1. 1. 1.]
two_S (angle/pi): [-0.5 -0.5 -0.5 -0.5]
two_S (after shift): [-0.5 -0.5 -0.5 -0.5]
S (sorted desc): [-0.25 -0.25 -0.25 -0.25]
n = round(sum(S)): -1  (sum(S) = -1.000000)
ones_mask: [0. 0. 0. 0.]
S after subtract: [-0.25 -0.25 -0.25 -0.25]
S after roll(--1): [-0.25 -0.25 -0.25 -0.25]
c (before folding): [-0.5 -0.5 -0.5]
FINAL: [1.5, 0.0, 0.49999999999999994]

=== CNOT ===
det(U_normalized): 1.000000+0.000000j
eigenvalues of gamma: [ 2.22044605e-16-1.j -1.38777878e-16+1.j  2.22044605e-16-1.j
 -1.38777878e-16+1.j]
|eigenvalues|: [1. 1. 1. 1.]
two_S (angle/pi): [-0.5  0.5 -0.5  0.5]
two_S (after shift): [-0.5  0.5 -0.5  0.5]
S (sorted desc): [ 0.25  0.25 -0.25 -0.25]
n = round(sum(S)): 0  (sum(S) = 0.000000)
ones_mask: [0. 0. 0. 0.]
S after subtract: [ 0.25  0.25 -0.25 -0.25]
S after roll(-0): [ 0.25  0.25 -0.25

In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
import jax.numpy as jnp
import numpy as np
from qiskit.circuit.library import SwapGate
from qiskit.quantum_info import Operator

from gulps.core.jax_invariants import _SYSY, _M, _su4_normalize

swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)

U = _su4_normalize(swap)
U_tilde = _SYSY @ U.T @ _SYSY
gamma = U @ U_tilde

ev = jnp.linalg.eigvals(gamma)
print("eigenvalues:", ev)
print("angles/pi:", jnp.angle(ev) / jnp.pi)

two_S = jnp.angle(ev) / jnp.pi
print("two_S:", two_S)
print("two_S == -0.5:", two_S == -0.5)
print("two_S <= -0.5:", two_S <= -0.5)
print("two_S < -0.5:", two_S < -0.5)

# Show the exact floating point values
for i, v in enumerate(two_S):
    print(f"  two_S[{i}] = {float(v):.20e}  (diff from -0.5: {float(v) - (-0.5):.2e})")

eigenvalues: [2.22044605e-16-1.j 2.22044605e-16-1.j 2.22044605e-16-1.j
 2.22044605e-16-1.j]
angles/pi: [-0.5 -0.5 -0.5 -0.5]
two_S: [-0.5 -0.5 -0.5 -0.5]
two_S == -0.5: [False False False False]
two_S <= -0.5: [False False False False]
two_S < -0.5: [False False False False]
  two_S[0] = -4.99999999999999944489e-01  (diff from -0.5: 5.55e-17)
  two_S[1] = -4.99999999999999944489e-01  (diff from -0.5: 5.55e-17)
  two_S[2] = -4.99999999999999944489e-01  (diff from -0.5: 5.55e-17)
  two_S[3] = -4.99999999999999944489e-01  (diff from -0.5: 5.55e-17)


In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
import jax.numpy as jnp
import numpy as np
from gulps.core.jax_invariants import _M

# Simulate what happens when two_S stays at -0.5 (not shifted)
two_S = jnp.array([-0.5, -0.5, -0.5, -0.5])
# No shift applied (the <= check fails)

S = jnp.sort(two_S / 2.0)[::-1]
print("S:", S)  # [-0.25, -0.25, -0.25, -0.25]

n = jnp.round(jnp.sum(S)).astype(jnp.int32)
print("n:", n, " sum(S):", jnp.sum(S))  # sum = -1, n = -1

ones_mask = (jnp.arange(4, dtype=jnp.int32) < n).astype(jnp.float64)
print("ones_mask:", ones_mask)  # n=-1, so no entries < -1, all 0

S_shifted = S - ones_mask
print("S after subtract:", S_shifted)

S_rolled = jnp.roll(S_shifted, -int(n))
print("S after roll:", S_rolled)

c = _M @ S_rolled[:3]
print("c before folding:", c)

c1, c2, c3 = float(c[0]), float(c[1]), float(c[2])
if c3 < 0:
    c1 = 1.0 - c1
    c3 = -c3
c2 = max(c2, 0.0)
c3 = max(c3, 0.0)
print(f"FINAL: [{c1}, {c2}, {c3}]")

print()
print("--- vs correct (shifted) path ---")
two_S_shifted = jnp.array([1.5, 1.5, 1.5, 1.5])
S2 = jnp.sort(two_S_shifted / 2.0)[::-1]
print("S:", S2)
n2 = jnp.round(jnp.sum(S2)).astype(jnp.int32)
print("n:", n2)
ones_mask2 = (jnp.arange(4, dtype=jnp.int32) < n2).astype(jnp.float64)
S_shifted2 = S2 - ones_mask2
S_rolled2 = jnp.roll(S_shifted2, -int(n2))
c2_correct = _M @ S_rolled2[:3]
print("c before folding:", c2_correct)
c1c, c2c, c3c = float(c2_correct[0]), float(c2_correct[1]), float(c2_correct[2])
if c3c < 0:
    c1c = 1.0 - c1c
    c3c = -c3c
c2c = max(c2c, 0.0)
c3c = max(c3c, 0.0)
print(f"FINAL: [{c1c}, {c2c}, {c3c}]")

S: [-0.25 -0.25 -0.25 -0.25]
n: -1  sum(S): -1.0
ones_mask: [0. 0. 0. 0.]
S after subtract: [-0.25 -0.25 -0.25 -0.25]
S after roll: [-0.25 -0.25 -0.25 -0.25]
c before folding: [-0.5 -0.5 -0.5]
FINAL: [1.5, 0.0, 0.5]

--- vs correct (shifted) path ---
S: [0.75 0.75 0.75 0.75]
n: 3
c before folding: [ 0.5  0.5 -0.5]
FINAL: [0.5, 0.5, 0.5]


In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
import jax.numpy as jnp
from qiskit.circuit.library import SwapGate, CXGate, iSwapGate
from qiskit.quantum_info import Operator, random_unitary

from gulps.core.jax_invariants import weyl_coordinates

for name, gate in [
    ("SWAP", SwapGate()),
    ("CNOT", CXGate()),
    ("iSWAP", iSwapGate()),
    ("sqrt_iSWAP", iSwapGate().power(1 / 2)),
    ("Identity", None),
]:
    if gate is None:
        U = jnp.eye(4, dtype=jnp.complex128)
    else:
        U = jnp.array(Operator(gate).data, dtype=jnp.complex128)
    w = weyl_coordinates(U)
    print(f"{name:15s}: {w}")

# Also check some random unitaries
for seed in range(5):
    U = jnp.array(random_unitary(4, seed=seed).data, dtype=jnp.complex128)
    w = weyl_coordinates(U)
    print(f"random(seed={seed}): {w}")

SWAP           : [0.5 0.5 0.5]
CNOT           : [5.00000000e-01 5.55111512e-17 5.55111512e-17]
iSWAP          : [0.5 0.5 0. ]
sqrt_iSWAP     : [0.25 0.25 0.  ]
Identity       : [0. 0. 0.]
random(seed=0): [0.36783129 0.25847022 0.16104988]
random(seed=1): [0.44451617 0.27017246 0.24178981]
random(seed=2): [0.81259404 0.05332171 0.03744055]
random(seed=3): [0.59871863 0.25569966 0.17422507]
random(seed=4): [0.44310672 0.17768403 0.04970487]


In [14]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
import jax
import jax.numpy as jnp
import numpy as np
from qiskit.circuit.library import CXGate, SwapGate, iSwapGate
from qiskit.quantum_info import Operator, random_unitary

from gulps.core.jax_invariants import weyl_coordinates, makhlin_invariants
from gulps.synthesis.jax_lm import _construct_unitary, _weyl_residual, NUM_PARAMS

print("NUM_PARAMS:", NUM_PARAMS)

# Reproduce fixture5: [CXGate(), iSwapGate().power(1/2), SwapGate()] costs [1.0, 0.5, 0.0]
# The SWAP basis gate case: prefix is some entangling unitary, basis is SWAP
swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)
sqiswap = jnp.array(Operator(iSwapGate().power(1 / 2)).data, dtype=jnp.complex128)

prefix = sqiswap  # simulate accumulated prefix
basis = swap
target_u = jnp.array(random_unitary(4, seed=0).data, dtype=jnp.complex128)
target_weyl = weyl_coordinates(target_u)
print("target_weyl:", target_weyl)

# Try running the LM solver once manually to see what happens
from jaxopt import LevenbergMarquardt
from jaxopt.linear_solve import solve_lu

weyl_solver = LevenbergMarquardt(
    residual_fun=_weyl_residual,
    maxiter=64,
    tol=1e-10,
    implicit_diff=False,
    jit=True,
    materialize_jac=True,
    solver=solve_lu,
)

# Random init
key = jax.random.PRNGKey(0)
init_params = jax.random.uniform(
    key, shape=(NUM_PARAMS,), minval=-jnp.pi / 2, maxval=jnp.pi / 2
)

# First check: what's the residual at init?
res0 = _weyl_residual(init_params, prefix, basis, target_weyl)
print("Residual at init:", res0)
print("Any NaN in residual:", jnp.any(jnp.isnan(res0)))

# Run the solver
result = weyl_solver.run(
    init_params, prefix_op=prefix, basis_gate=basis, target_inv=target_weyl
)
print()
print("LM result params:", result.params)
print("LM result state.residual:", result.state.residual)
print("Any NaN in LM params:", jnp.any(jnp.isnan(result.params)))
print("Any NaN in LM residual:", jnp.any(jnp.isnan(result.state.residual)))
print("Max abs residual:", jnp.max(jnp.abs(result.state.residual)))

# Check the Jacobian
jac = jax.jacobian(lambda p: _weyl_residual(p, prefix, basis, target_weyl))(init_params)
print()
print("Weyl Jacobian at init:")
print("  rank:", np.linalg.matrix_rank(np.array(jac), tol=1e-8))
print("  svs:", np.linalg.svd(np.array(jac), compute_uv=False))
print("  any NaN:", np.any(np.isnan(np.array(jac))))

NUM_PARAMS: 8
target_weyl: [0.36783129 0.25847022 0.16104988]
Residual at init: [-0.13216871  0.00847022 -0.08895012]
Any NaN in residual: False

LM result params: [-4.53197877e+14 -7.93343470e+14  6.10204489e+12 -2.41921963e+14
 -3.74561866e+13 -8.52424317e+12 -6.80023314e+14  8.86156016e+14]
LM result state.residual: [-0.13216871  0.00847022 -0.08895012]
Any NaN in LM params: False
Any NaN in LM residual: False
Max abs residual: 0.13216870918092938

Weyl Jacobian at init:
  rank: 0
  svs: [2.19470191e-16 5.96465623e-17 4.95253185e-17]
  any NaN: False


In [15]:
import os

os.environ["JAX_ENABLE_X64"] = "1"
import jax
import jax.numpy as jnp
import numpy as np
from qiskit.circuit.library import CXGate, SwapGate, iSwapGate
from qiskit.quantum_info import Operator, random_unitary

from gulps.core.jax_invariants import weyl_coordinates, makhlin_invariants
from gulps.synthesis.jax_lm import (
    _construct_unitary,
    _weyl_residual,
    _makhlin_residual,
    NUM_PARAMS,
)
from gulps.synthesis.jax_lm import _make_restart_loop, _make_warmstart_loop
from jaxopt import GaussNewton, LevenbergMarquardt
from jaxopt.linear_solve import solve_lu
from jax.random import PRNGKey

print("NUM_PARAMS:", NUM_PARAMS)

# Reproduce fixture5: ISA includes SWAP as basis gate
swap = jnp.array(Operator(SwapGate()).data, dtype=jnp.complex128)
sqiswap = jnp.array(Operator(iSwapGate().power(1 / 2)).data, dtype=jnp.complex128)
cx = jnp.array(Operator(CXGate()).data, dtype=jnp.complex128)

# Try several prefix + SWAP combos
for name, prefix in [
    ("sqiswap", sqiswap),
    ("cx", cx),
    ("identity", jnp.eye(4, dtype=jnp.complex128)),
]:
    basis = swap
    target_u = jnp.array(random_unitary(4, seed=0).data, dtype=jnp.complex128)
    target_weyl = weyl_coordinates(target_u)
    target_makhlin = makhlin_invariants(target_u)

    # Stage 1: solve Makhlin
    makhlin_solver = GaussNewton(
        residual_fun=_makhlin_residual,
        maxiter=256,
        tol=1e-10,
        implicit_diff=False,
        implicit_diff_solve=solve_lu,
        jit=True,
    )
    solve_makhlin = _make_restart_loop(makhlin_solver.run, 256)

    key = PRNGKey(0)
    makhlin_params, makhlin_res = solve_makhlin(
        key, prefix, basis, target_makhlin, 1e-9
    )

    # Check warm start Weyl residual
    warm_start_res = _weyl_residual(makhlin_params, prefix, basis, target_weyl)
    warm_res_val = jnp.max(jnp.abs(warm_start_res))

    # Check the constructed unitary's Weyl coords
    U = _construct_unitary(makhlin_params, prefix, basis)
    constructed_weyl = weyl_coordinates(U)

    print(f"prefix={name}:")
    print(f"  Stage 1 res: {float(makhlin_res):.2e}")
    print(f"  constructed weyl: {constructed_weyl}")
    print(f"  target weyl:      {target_weyl}")
    print(f"  warm start Weyl residual: {warm_start_res}")
    print(f"  max |weyl res|: {float(warm_res_val):.2e}")
    print(f"  any NaN: {bool(jnp.any(jnp.isnan(warm_start_res)))}")
    print()

NUM_PARAMS: 8
prefix=sqiswap:
  Stage 1 res: 8.02e-01
  constructed weyl: [nan nan nan]
  target weyl:      [0.36783129 0.25847022 0.16104988]
  warm start Weyl residual: [nan nan nan]
  max |weyl res|: nan
  any NaN: True

prefix=cx:
  Stage 1 res: 8.02e-01
  constructed weyl: [nan nan nan]
  target weyl:      [0.36783129 0.25847022 0.16104988]
  warm start Weyl residual: [nan nan nan]
  max |weyl res|: nan
  any NaN: True

prefix=identity:
  Stage 1 res: 2.80e+00
  constructed weyl: [nan nan nan]
  target weyl:      [0.36783129 0.25847022 0.16104988]
  warm start Weyl residual: [nan nan nan]
  max |weyl res|: nan
  any NaN: True

